# Log-gamma sparse-table experiments

This notebook is a lightweight public-facing entry point.  The reusable code lives in `src/gamma_sparse_table`, and larger experiments should be run from `scripts/`.


In [ ]:
from pathlib import Path

import pandas as pd

from gamma_sparse_table.core import FitOptions, create_problem, default_gamma_grid
from gamma_sparse_table.experiments import run_high_leverage_stress
from gamma_sparse_table.plotting import plot_method_bar, summarize_wj_curve, plot_wj_mse_curve


## Create the default problem

The default problem has ten binary factors, all main effects and all pairwise interactions, and a sparse clean parameter.


In [ ]:
problem = create_problem(q=10, n=3000)
print(f"K = {problem.K}, d = {problem.d}, active = {len(problem.active)}")
pd.Series(problem.n * problem.p_clean).describe()


## Quick high-leverage stress run

This small run is only a smoke test.  For manuscript-level Monte Carlo accuracy, use `scripts/run_high_leverage_stress.py --n-reps 60 --b-resample 10 --n-jobs 7`.


In [ ]:
out_dir = Path("../results/notebook_high_leverage_stress")
gamma_grid = default_gamma_grid(max_gamma=2.0)
results, curves, summary = run_high_leverage_stress(
    problem=problem,
    out_dir=out_dir,
    n_reps=2,
    n_jobs=1,
    gamma_grid=gamma_grid,
    b_resample=2,
    options=FitOptions(ridge=1e-4),
)
summary


In [ ]:
curve_summary = summarize_wj_curve(curves)
plot_method_bar(summary, "rmse_active_mean", "rmse_active_se", "Active-parameter RMSE", out_dir / "active_rmse.png")
plot_wj_mse_curve(curve_summary, out_dir / "wj_mse_curve.png")
print(f"Saved outputs in {out_dir}")
